# Predicción y Determinantes de la Pobreza Multidimensional y Vulnerabilidad
### CASEN 2024 — Taller de Programación, UBA 2026 — Grupo 2

Notebook de arranque: carga de datos, cruce con la base de provincia/comuna, definición de la unidad de análisis y primera limpieza. Pensado para correr en **Google Colab**.


## 1. Setup

In [1]:
!pip install pyreadstat -q
import pandas as pd
import numpy as np
import pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.7 MB/s eta 0:00:00


## 2. Cargar los datos Google Drive (carpeta `CASEN2024/`):
- `casen_2024stata.zip` → contiene la base principal `casen_2024.dta` (877 variables), **comprimida**
- `casen_2024_provincia_comuna.dta` → base chica de provincia/comuna, ya viene suelta (no hace falta descomprimir)

La base principal viene **zipeada**, así que primero hay que descomprimirla antes de leerla con `pyreadstat`.

In [2]:
import zipfile

from google.colab import drive
drive.mount('/content/drive')

# Ajustar el path según donde hayas guardado los archivos en tu Drive
RUTA = '/content/drive/MyDrive/CASEN2024/'

# Descomprimir la base principal (viene dentro de casen_2024stata.zip)
with zipfile.ZipFile(RUTA + 'casen_2024stata.zip') as z:
    z.extractall(RUTA)  # extrae casen_2024.dta en la misma carpeta

# Base principal (877 variables)
df, meta = pyreadstat.read_dta(RUTA + 'casen_2024.dta')

# Base complementaria de provincia y comuna (folio, id_persona, provincia, comuna, expp, expc) — ya viene suelta
df_geo, meta_geo = pyreadstat.read_dta(RUTA + 'casen_2024_provincia_comuna.dta')

print(df.shape, df_geo.shape)

Mounted at /content/drive
(218367, 877) (218367, 6)


## 3. Cruzar ambas bases

Según la *Nota de uso de bases de datos Casen 2024*, la llave de cruce es **folio + id_persona**.

In [3]:
df = df.merge(df_geo[['folio', 'id_persona', 'provincia', 'comuna', 'expp', 'expc']],
              on=['folio', 'id_persona'], how='left')
df.shape

(218367, 881)

## 4. Definir la unidad de análisis: hogar

Trabajamos a nivel de **hogar**, usando `folio` como identificador y filtrando por jefatura de hogar (`pco1_a == 1`) para no repetir observaciones dentro de un mismo hogar.

In [4]:
hogares = df[df['pco1_a'] == 1].copy()
print('Hogares:', hogares.shape[0])

Hogares: 78654


## 5. Variable objetivo

`pobreza_multi`: 0 = hogar fuera de pobreza multidimensional, 1 = hogar en pobreza multidimensional.

In [5]:
hogares['pobreza_multi'].value_counts(dropna=False)
tasa = hogares['pobreza_multi'].mean() * 100
print(f'Tasa de pobreza multidimensional (sin ponderar): {tasa:.1f}%')

# Con factor de expansión regional (representativo a nivel nacional)
tasa_expandida = np.average(hogares['pobreza_multi'], weights=hogares['expr'])
print(f'Tasa de pobreza multidimensional (ponderada, expr): {tasa_expandida*100:.1f}%')

Tasa de pobreza multidimensional (sin ponderar): 13.6%
Tasa de pobreza multidimensional (ponderada, expr): nan%


## 6. Selección de variables predictoras (no monetarias)

⚠️ **No usamos las variables `hh_d_*`** (carencias) como predictoras porque son los componentes que construyen `pobreza_multi` — usarlas sería circular. Partimos de las variables crudas de los módulos S (Salud), E (Educación) y V (Vivienda).

In [6]:
predictores_salud = ['s13', 's16', 's17', 's19a', 's19b', 's19c', 's19d', 's19e', 's2']
predictores_educacion = ['educ']  # crear a partir de CH12/CH13/CH14 si no viene calculada; revisar libro de códigos módulo E
predictores_vivienda = ['v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v12', 'v13']

predictores = predictores_salud + predictores_educacion + predictores_vivienda
predictores = [c for c in predictores if c in hogares.columns]
print('Predictores disponibles:', predictores)

base_modelo = hogares[['folio', 'provincia', 'comuna', 'expr', 'pobreza_multi'] + predictores].copy()
base_modelo.head()

Predictores disponibles: ['s13', 's16', 's17', 's19a', 's19b', 's19c', 's19d', 's19e', 's2', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v12', 'v13']


,folio,provincia,comuna,expr,pobreza_multi,s13,s16,s17,s19a,s19b,...,s19e,s2,v2,v3,v4,v5,v6,v7,v12,v13
0,100020301,131,13127,264,0,1,4,2,NaN,NaN,...,NaN,NaN,2,2,1,1,3,2,3,1
2,100020401,131,13127,295,0,1,5,NaN,NaN,NaN,...,NaN,NaN,2,1,1,1,3,1,4,2
6,100020501,131,13127,373,0,1,2,1,2,2,...,2,NaN,3,3,5,3,3,3,3,3
7,100020801,131,13127,266,0,1,5,NaN,NaN,NaN,...,NaN,NaN,2,2,1,1,3,1,4,1
12,100070201,22,2203,72,0,1,5,NaN,NaN,NaN,...,NaN,NaN,2,1,2,1,3,1,2,1


## 7. Limpieza: valores perdidos y códigos especiales

Revisar los códigos de "No sabe/No responde" del libro de códigos para cada variable (suelen venir como 88, 99, -88, -99 según el módulo) y decidir imputación o exclusión antes de modelar.

In [7]:
base_modelo.isna().sum().sort_values(ascending=False)

,0
s2,78654
s19a,62705
s19b,62542
s19c,62542
s19e,62542
s19d,62542
s17,60943
pobreza_multi,1889
s13,0
provincia,0


## 8. Guardar la base limpia

Para no volver a levantar los archivos pesados (.dta/.sav) en cada corrida, guardamos un CSV liviano con las variables ya seleccionadas — este es el archivo que subimos al repo de GitHub (los .dta/.sav originales NO se suben, van en `.gitignore`).

In [8]:
base_modelo.to_csv('base_modelo_casen2024.csv', index=False)
from google.colab import files
files.download('base_modelo_casen2024.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Próximos pasos (Aplicación Final)

- Completar la limpieza (recodificar NS/NR, crear variable `educ` si hace falta)
- Estadística descriptiva de los predictores por grupo (pobre / no pobre)
- Modelos: logit/probit (interpretabilidad) + Random Forest o XGBoost (poder predictivo) con validación train/test
- Comparar importancia de variables entre dimensiones (salud, educación, entorno)
